In [1]:
import polars as pl
import seaborn as sns
import iggnition

## Pre-flight

In [2]:
# Load file / antibody library (paired data)
file = "./20260309_complete_database_with_lineages_and_subsets.parquet"
df = pl.read_parquet(file)

In [3]:
print(f"Total = {df.height:2,d} paired sequences")

Total = 4,840,405 paired sequences


In [4]:
# Filter out naive B cells (IgM/D with less than 1% mutation [approximated in the V gene of heavy chain])
df = df.filter(
    ~((pl.col('v_identity:0') >= 0.99) & (pl.col('c_gene:0') == 'IGHM'))
)

In [5]:
print(f"Total = {df.height:2,d} memory B cells")

Total = 2,448,194 memory B cells


In [11]:
mutated, _ = iggnition.run(df, name_col="name", wide=True, verbose=True, human_readable=False)

numbering:   0%|          | 0/4896388 [00:00<?, ? seq/s]

In [12]:
germline, _ = iggnition.run(df, paired=True, 
                            nt_col_heavy="germline:0", aa_col_heavy="germline_aa:0", 
                            nt_col_light="germline:1", aa_col_light="germline_aa:1", 
                            name_col="name", wide=True, verbose=True, human_readable=False)

numbering:   0%|          | 0/4896388 [00:00<?, ? seq/s]

## Step 1: List mutations

In [18]:
mutated

seq_name,H1,H2,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12,H13,H14,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26,H27,H28,H29,H30,H31,H32,H33,H34,H35,H36,…,L408,L409,L410,L411,L412,L413,L414,L415,L416,L417,L418,L419,L420,L421,L422,L423,L424,L425,L426,L427,L428,L429,L430,L431,L432,L433,L434,L435,L436,L437,L438,L439,L440,L441,L442,L443,L444
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""ACGATGTTCCCTAACC_d01_w83""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,45,45,45,45,45,45,45,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""TCGAGGCTCTTTCCTC_d01_w8""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,71,65,84,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""GCGCAGTTCAGCTTAG_d01_w68""",67,65,71,71,84,67,67,65,65,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,67,67,84,71,65,65,65,84,71,…,45,45,45,45,45,45,45,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""GAAATGAGTGGTACAG_d01_w74""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,71,65,84,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""TTTCCTCTCATGCTCC_d01_w83""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,71,65,84,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ATCTACTTCGACAGCC_d11_w177""",71,65,71,71,84,71,67,65,71,71,84,71,71,84,71,71,65,65,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,67,84,71,…,45,45,45,45,45,45,45,71,65,71,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""TACGGATAGAGTGAGA_d02_w187""",67,65,71,71,84,71,67,65,71,67,84,71,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,65,67,67,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""GGTGCGTTCACTTACT_d09_w167""",67,65,71,71,84,71,67,65,71,67,84,71,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,65,67,67,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45


In [14]:
germline

seq_name,H1,H2,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12,H13,H14,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26,H27,H28,H29,H30,H31,H32,H33,H34,H35,H36,…,L408,L409,L410,L411,L412,L413,L414,L415,L416,L417,L418,L419,L420,L421,L422,L423,L424,L425,L426,L427,L428,L429,L430,L431,L432,L433,L434,L435,L436,L437,L438,L439,L440,L441,L442,L443,L444
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""ACGATGTTCCCTAACC_d01_w83""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,45,45,45,45,45,45,45,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""TCGAGGCTCTTTCCTC_d01_w8""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,71,65,84,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""GCGCAGTTCAGCTTAG_d01_w68""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""GAAATGAGTGGTACAG_d01_w74""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,71,65,84,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""TTTCCTCTCATGCTCC_d01_w83""",67,65,71,71,84,67,67,65,71,67,84,84,71,84,71,67,65,71,84,67,84,45,45,45,71,71,71,71,67,84,71,65,71,71,84,71,…,45,45,45,45,45,45,45,71,65,84,65,84,67,65,65,65,67,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GGGAGATTCCCTTGTG_d05_w148""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GACGTTATCAAACCGT_d09_w107""",71,65,71,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,65,71,71,65,71,71,67,84,84,71,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""CTCCTAGCAAAGTCAA_d04_w112""",67,65,71,71,84,84,67,65,71,67,84,71,71,84,71,67,65,71,84,67,84,45,45,45,71,71,65,71,67,84,71,65,71,71,84,71,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [24]:
key = "seq_name"
value_cols = [c for c in mutated.columns if c != key]

mutations = (
    mutated
    .join(germline, on=key, how="left", suffix="_2")
    .with_columns([
        (pl.col(c) - pl.col(f"{c}_2")).alias(c)
        for c in value_cols
    ])
    .select([key] + value_cols)
)

In [25]:
mutations

seq_name,H1,H2,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12,H13,H14,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26,H27,H28,H29,H30,H31,H32,H33,H34,H35,H36,…,L408,L409,L410,L411,L412,L413,L414,L415,L416,L417,L418,L419,L420,L421,L422,L423,L424,L425,L426,L427,L428,L429,L430,L431,L432,L433,L434,L435,L436,L437,L438,L439,L440,L441,L442,L443,L444
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""ACGATGTTCCCTAACC_d01_w83""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""TCGAGGCTCTTTCCTC_d01_w8""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""GCGCAGTTCAGCTTAG_d01_w68""",0,0,0,0,0,0,0,0,250,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,252,0,0,0,0,250,250,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""GAAATGAGTGGTACAG_d01_w74""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""TTTCCTCTCATGCTCC_d01_w83""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ATCTACTTCGACAGCC_d11_w177""",0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,250,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""TACGGATAGAGTGAGA_d02_w187""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""GGTGCGTTCACTTACT_d09_w167""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Step 2: Find motifs